In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

# 1.消息类型

在LangChain中，发送给LLM的消息、LLM返回的消息都统一被封装为BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage：role是tool，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息。

In [3]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# 定义工具
@tool
def get_weather(location: str) -> str:
    """
    Get the weather in a given location.
    Args:
        location: city name or coordinates
    """
    return f"Current weather in {location} is sunny"

# 创建Agent
agent = create_agent(model="deepseek-chat", tools=[get_weather])

# 调用Agent，发送消息
response = agent.invoke({
    "messages": [
        SystemMessage("请使用工具来获取天气信息。"),
        HumanMessage("你好，我是虎哥."),
        AIMessage("你好，虎哥，很高兴认识你."),
        HumanMessage("北京今天天气如何？")
    ]
})

print(response["messages"][-1].content)

虎哥，北京今天天气晴朗（sunny）。如果你要出门，我可以帮你查更多细节，不过当前工具只返回了天气状况这一项信息。


In [4]:
for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

请使用工具来获取天气信息。
================================ Human Message =================================

你好，我是虎哥.
================================== Ai Message ==================================

你好，虎哥，很高兴认识你.
================================ Human Message =================================

北京今天天气如何？
================================== Ai Message ==================================

I'll check the weather in Beijing for you.
Tool Calls:
  get_weather (call_00_nhnMQtdJdCDnr3JlrR9O7358)
 Call ID: call_00_nhnMQtdJdCDnr3JlrR9O7358
  Args:
    location: 北京
================================= Tool Message =================================
Name: get_weather

Current weather in 北京 is sunny
================================== Ai Message ==================================

虎哥，北京今天天气晴朗（sunny）。如果你要出门，我可以帮你查更多细节，不过当前工具只返回了天气状况这一项信息。


# 2.多模态消息

之前我们都是向模型发送文本消息，但是 LangChain 也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：
- qwen3.5-plus
- gpt-5-nano
- ...

我们以qwen3.5-plus为例，演示向模型发送图片消息

## 2.1.在线图片
首先，我们演示如何发送一个在线图片给模型，也就是指定模型的url地址。
图片如下：

<img src="https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg" width="500" height="300" alt="图片描述">



In [3]:
from langchain.chat_models import init_chat_model
import os

# 初始化模型
model = init_chat_model(
    model="qwen3.5-plus",  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

In [6]:
# 创建Agent
agent = create_agent(model=model)

In [7]:
# 准备多模态消息
message = HumanMessage([
        {"type": "text", "text": "描述以下这张图片的内容."},
        {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},
    ])

In [8]:
stream = agent.stream(
    {"messages": [message]},
    stream_mode="messages"
)
for chunk, metadata in stream:
    if chunk.content:
        print(chunk.content, end="", flush=True)

这张图片捕捉了一个非常温馨、宁静的时刻，以下是详细的内容描述：

**1. 主体人物与动物：**
*   **女性：** 一位年轻女性侧身坐在沙滩上。她留着深色长发，身穿一件黑白格纹的长袖衬衫和深色长裤（裤脚似乎卷起）。她的脸上洋溢着灿烂、幸福的笑容，正温柔地注视着面前的狗。她的左手腕上戴着一块白色的手表。
*   **狗：** 一只浅黄色的拉布拉多犬（或金毛寻回犬）乖巧地坐在女子对面。它身上佩戴着带有彩色图案的胸背带（harness），一条红色的牵引绳散落在它身后的沙地上。

**2. 动作与互动：**
*   这是画面的核心：狗狗抬起它的右前爪，轻轻地放在女子的手掌中，仿佛正在执行“握手”的指令。女子也伸出手掌回应，两人（人和狗）之间正在进行亲密的互动，展现了深厚的感情。

**3. 环境与背景：**
*   **地点：** 场景设定在宽阔的海滩上，前景是细腻的沙地，上面布满了自然的纹理和脚印。
*   **背景：** 远处是平静的大海，可以看到白色的浪花正在轻轻拍打着海岸线。
*   **光线与氛围：** 图片右侧有强烈的阳光照射进来（看起来像是日落时分的夕阳），给画面镀上了一层温暖的金黄色调。这种逆光效果让女子的发丝和轮廓显得柔和，营造出一种温暖、治愈且充满爱意的氛围。

## 2.2.本地图片数据
有时候用户会上传图片数据，而不是图片的url地址。我们需要将图片数据转换成base64字符串，然后发送给模型。

接下来我们会模拟图片上传、转换的过程。

首先，我们安装一个上传组件，用于模拟图片上传。


In [ ]:
!uv add ipywidgets

然后，我们创建一个上传组件，用于模拟图片上传。


In [5]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*', multiple=False)
display(uploader)

FileUpload(value=(), accept='*', description='Upload')

In [6]:
print(uploader.value)

()


In [7]:
# 读取图片，转为base64字符串
import base64

# 获取第一个（也是唯一一个）上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file["content"]

# 转换内存视图->字节
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

In [8]:
# 组织多模态消息
multimodal_question = HumanMessage(content=[
    {
        "type": "image",
        "base64": img_b64,
        "mime_type": "image/jpeg",
    },
    {"type": "text", "text": "给我讲讲图片中的城市"}
])

for chunk, metadata in agent.stream(
    {"messages": [multimodal_question]},
    stream_mode="messages"
):
    print(chunk.content, end="", flush=True)

图片中并没有直接写明城市名称，但**图片右下角的水印**给出了关键线索——**南昌大学（Nanchang University）**的校徽。因此，这个场景大概率拍摄于**江西省南昌市**。

结合图片内容，可以解读出以下城市信息：

1. **地标与背景**：广场上有高大的棕榈科植物，这在我国南方城市很常见。后方空旷、开阔的铺装广场结合大学校徽，说明这很可能是南昌大学某校区的广场。
2. **核心雕塑**：中央的红色大型雕塑呈现了“70”字样，右侧写着“**不忘初心 牢记使命**”，下方有一行小字：“**热烈庆祝中华人民共和国成立70周年 1949-2019**”，雕塑上还有和平鸽造型。这说明该照片拍摄于**2019年前后**，是国庆70周年的主题景观装置。
3. **城市气质**：南昌是“英雄城”（八一南昌起义发生地），也是一座红色文化底蕴深厚的城市，这类红色主题的公共艺术装置在当地高校、广场非常普遍。南昌大学作为江西唯一的“211工程”及“双一流”高校，是南昌的重要科教名片。

**关于南昌这座城市，补充介绍：**
* **位置**：江西省省会，地处赣江、抚河下游，鄱阳湖西南岸。
* **历史文化**：拥有滕王阁（江南三大名楼之一）、八一起义纪念馆等名胜，也是“军旗升起的地方”。
* **特色**：有“物华天宝、人杰地灵”之称，城市气质兼具红色历史与现代湖光水色（如青山湖、艾溪湖）。

如果你想确认具体是南昌大学的哪个校区，或者想了解南昌的旅游、天气等信息，可以告诉我！